In [ ]:
path_to_dataset = "" #FIXME: Add path to dataset here

In [ ]:
import os
import cv2
import torch
import itertools
import pandas as pd
from PIL import Image
from sklearn.metrics import precision_score, recall_score, fbeta_score

import sys
sys.path.append('..')

from Pipeline import Pipeline

In [ ]:
# Initialize the pipeline
pipeline = Pipeline()

csv_path = "../Training_data/notch_classification_dataset.csv"

df = pd.read_csv(csv_path)
val_df = df[df['split'] == 'test']

image_files = val_df['image_filename'].unique().tolist()
print(f"Found {len(image_files)} validation images in the CSV.")

In [ ]:
def calculate_iou(boxA, boxB):
    """Calculates Intersection over Union for two bounding boxes."""
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[2], boxB[2])
    yB = min(boxA[3], boxB[3])

    interArea = max(0, xB - xA) * max(0, yB - yA)
    boxAArea = (boxA[2] - boxA[0]) * (boxA[3] - boxA[1])
    boxBArea = (boxB[2] - boxB[0]) * (boxB[3] - boxB[1])

    iou = interArea / float(boxAArea + boxBArea - interArea + 1e-5)
    return iou

def get_ground_truth_boxes(df, image_filename, img_w, img_h):
    """Parses normalized CSV coordinates into absolute bounding boxes."""
    boxes = []
    # Filter the dataframe for just this image
    img_data = df[df['image_filename'] == image_filename]
    
    for _, row in img_data.iterrows():
        # Treat anything that ISN'T noise as a target notch we want to detect
        if row['super_category'].lower() != 'noise':
            cx, cy, w, h = row['x_center'], row['y_center'], row['width'], row['height']
            
            # Convert normalized YOLO format (0-1) to absolute pixel coordinates
            x1 = int((cx - w/2) * img_w)
            y1 = int((cy - h/2) * img_h)
            x2 = int((cx + w/2) * img_w)
            y2 = int((cy + h/2) * img_h)
            
            boxes.append((x1, y1, x2, y2))
            
    return boxes

In [ ]:
def build_pipeline_cache(pipeline, val_df, images_dir, base_yolo_conf=0.01):
    """
    Runs the heavy parts of the pipeline (YOLO + DINOv2 + SVM) once at the 
    lowest thresholds, caching the probabilities so the grid search is instant.
    """
    image_files = val_df['image_filename'].unique().tolist()
    print(f"Caching predictions for {len(image_files)} images (Base YOLO conf: {base_yolo_conf})...")
    cache = {}
    
    for img_file in image_files:
        subfolder = img_file[0].upper()
        img_path = os.path.join(images_dir, subfolder, img_file)

        img = cv2.imread(img_path)
        if img is None: 
            print(f"Warning: Could not read {img_path}")
            continue

        h, w = img.shape[:2]
        
        # 1. Get Ground Truth
        gt_boxes = get_ground_truth_boxes(val_df, img_file, w, h)
        
        # 2. Check for Border (mirrors process_image)
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if len(img.shape) == 3 else img
        hist = pipeline.get_perimeter_histogram(gray, band_width=15)
        is_border = pipeline.border_classification(hist, threshold=40, ratio=0.84)
        
        processed_candidates = []
        
        # 3. Only run YOLO & DINO if a border is present
        if is_border:
            raw_candidates = pipeline.get_yolo_candidates(img, yolo_conf=base_yolo_conf)
            film_bbox = pipeline._get_film_bounding_box(img, strip_tolerance=0.02, patch_size=15)
            
            for box in raw_candidates:
                crop, _ = pipeline._extract_and_align_notch(img, box['coords'], film_bbox, padding=10)
                if crop.size == 0: continue
                    
                # Extract DINOv2 features
                pil_crop = Image.fromarray(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
                img_tensor = pipeline.transform(pil_crop).unsqueeze(0).to(pipeline.device)
                
                with torch.no_grad():
                    feature_vector = pipeline.dinov2(img_tensor).cpu().numpy().flatten()
                    
                # Get SVM probability for Notch vs Noise
                notch_idx = list(pipeline.svm_notch_noise.classes_).index('notch')
                prob_notch = pipeline.svm_notch_noise.predict_proba([feature_vector])[0][notch_idx]
                
                processed_candidates.append({
                    'coords': box['coords'],
                    'yolo_prob': box['prob'],
                    'svm_notch_prob': prob_notch
                })
                
        cache[img_file] = {
            'candidates': processed_candidates,
            'ground_truth': gt_boxes
        }
        
    print("Caching complete!")
    return cache

In [ ]:
# Build the cache (Takes a bit of time, but you only do it once)
cache = build_pipeline_cache(pipeline, val_df, path_to_dataset, base_yolo_conf=0.01)

In [ ]:
def evaluate_yolo_only(yolo_options, cache):
    """Baseline: Evaluates performance using ONLY YOLO confidence thresholds."""
    all_results = []
    for y_conf in yolo_options:
        y_true, y_pred = [], []
        
        for data in cache.values():
            gt_boxes = data['ground_truth']
            # Ignore SVM, filter purely by YOLO prob
            yolo_boxes = [c['coords'] for c in data['candidates'] if c['yolo_prob'] >= y_conf]
            
            matched_gt = set()
            for p_box in yolo_boxes:
                match_found = False
                for i, gt_box in enumerate(gt_boxes):
                    if i not in matched_gt and calculate_iou(p_box, gt_box) > 0.35:
                        matched_gt.add(i)
                        y_true.append(1); y_pred.append(1)
                        match_found = True
                        break
                if not match_found:
                    y_true.append(0); y_pred.append(1) # False Positive
                    
            # False Negatives
            fn_count = len(gt_boxes) - len(matched_gt)
            y_true.extend([1] * fn_count)
            y_pred.extend([0] * fn_count)

        if not y_true: continue
            
        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)
        f1 = fbeta_score(y_true, y_pred, beta=1, zero_division=0)
        f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)
        
        all_results.append({'yolo_conf': y_conf, 'recall': r, 'precision': p, 'f2_score': f2, 'f1_score': f1})
        
    return pd.DataFrame(all_results)

def run_pipeline_grid_search(yolo_options, svm_options, cache):
    """Grid Search: Evaluates performance using BOTH YOLO and SVM filters."""
    best_f2 = 0
    best_params = {}
    all_results = []
    combinations = list(itertools.product(yolo_options, svm_options))
    print(f"Testing {len(combinations)} parameter combinations...\n")
    
    for y_conf, s_thresh in combinations:
        y_true, y_pred = [], []

        for data in cache.values():
            gt_boxes = data['ground_truth']
            
            final_boxes = [
                c['coords'] for c in data['candidates'] 
                if c['yolo_prob'] >= y_conf and c['svm_notch_prob'] >= s_thresh
            ]
            
            matched_gt = set()
            for p_box in final_boxes:
                match_found = False
                for i, gt_box in enumerate(gt_boxes):
                    if i not in matched_gt and calculate_iou(p_box, gt_box) > 0.35:
                        matched_gt.add(i)
                        y_true.append(1); y_pred.append(1)
                        match_found = True
                        break
                        
                if not match_found:
                    y_true.append(0); y_pred.append(1) # False Positive
                    
            # False Negatives
            fn_count = len(gt_boxes) - len(matched_gt)
            y_true.extend([1] * fn_count)
            y_pred.extend([0] * fn_count)
            
        if not y_true: continue
            
        p = precision_score(y_true, y_pred, zero_division=0)
        r = recall_score(y_true, y_pred, zero_division=0)
        f1 = fbeta_score(y_true, y_pred, beta=1, zero_division=0)
        f2 = fbeta_score(y_true, y_pred, beta=2, zero_division=0)

        all_results.append({
            'yolo_conf': y_conf, 'svm_threshold': s_thresh,
            'f2': f2, 'recall': r, 'precision': p, 'f1': f1
        })
        
        if f2 >= best_f2:
            best_f2 = f2
            best_params = {'yolo_conf': y_conf, 'svm_threshold': s_thresh, 'f2': f2, 'r': r, 'p': p, 'f1': f1}
            
    return pd.DataFrame(all_results), best_params

In [ ]:
# Define search ranges
yolo_options = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
svm_options  = [0.01, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.35, 0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]

print("--- EVALUATING YOLO BASELINE (No SVM) ---")
df_yolo_only = evaluate_yolo_only(yolo_options, cache)
display(df_yolo_only.sort_values(by='f2_score', ascending=False).head(5))

print("\n--- RUNNING FULL PIPELINE GRID SEARCH ---")
df_results, best = run_pipeline_grid_search(yolo_options, svm_options, cache)

# Save results
df_results.to_csv('pipeline_hyperparameter_sweep.csv', index=False)

print("\n" + "="*50)
print(f"🏆 BEST PIPELINE SETUP: YOLO = {best['yolo_conf']}, SVM = {best['svm_threshold']}")
print(f"F2-Score:  {best['f2']:.3f}")
print(f"Recall:    {best['r']:.3f} (Notches found)")
print(f"Precision: {best['p']:.3f} (Noise filtered out)")
print(f"F1-Score:  {best['f1']:.3f}")
print("="*50)

**Visualise sweep results**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns
import numpy as np

# Load the data
df_full = pd.read_csv('pipeline_hyperparameter_sweep.csv')

# 1. Heatmap for F2 Score
pivot_f2 = df_full.pivot(index='yolo_conf', columns='svm_threshold', values='f2')
plt.figure(figsize=(10, 8))
heatmap = sns.heatmap(pivot_f2, annot=True, cmap='viridis', fmt='.2f')
heatmap.invert_yaxis()
plt.title('Heatmap of F2 Score (YOLO vs SVM Thresholds)')
plt.ylabel('YOLO Confidence Threshold')
plt.xlabel('SVM Confidence Threshold')
plt.show()

# 2. 3D Surface Plot for F2 Score
X = pivot_f2.columns.values
Y = pivot_f2.index.values
X_mesh, Y_mesh = np.meshgrid(X, Y)
Z = pivot_f2.values

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')
surf = ax.plot_surface(X_mesh, Y_mesh, Z, cmap='viridis', edgecolor='none', alpha=0.8)
ax.invert_xaxis()
ax.set_xlabel('SVM Threshold')
ax.set_ylabel('YOLO Threshold')
ax.set_zlabel('F2 Score')
ax.set_title('3D Optimization Landscape: F2 Score')
fig.colorbar(surf, ax=ax, shrink=0.5, aspect=5)
plt.show()

# Create subplots
fig, axes = plt.subplots(1, 3, figsize=(24, 7))

metrics = ['recall', 'precision', 'f2']
cmaps = ['viridis', 'viridis', 'viridis']
titles = ['Recall (Sensitivity)', 'Precision (Specificity)', 'F2 Score (Optimized)']

for i, metric in enumerate(metrics):
    pivot = df_full.pivot(index='yolo_conf', columns='svm_threshold', values=metric)
    
    # Plot heatmap
    sns.heatmap(pivot, annot=True, cmap=cmaps[i], fmt='.2f', ax=axes[i])

    axes[i].invert_yaxis()
    
    # Locate the cell for the selected values
    y_idx = np.where(pivot.index == best['yolo_conf'])[0][0]
    x_idx = np.where(pivot.columns == best['svm_threshold'])[0][0]
    
    # Highlight the chosen cell with a thick cyan border
    axes[i].add_patch(Rectangle((x_idx, y_idx), 1, 1, fill=False, edgecolor='cyan', lw=4))
    
    axes[i].set_title(titles[i], fontsize=16, fontweight='bold')
    axes[i].set_ylabel('YOLO Threshold' if i==0 else '')
    axes[i].set_xlabel('SVM Threshold')

plt.suptitle(f"Parameter Sweep Results: Highlighted Choice ($YOLO={best['yolo_conf']}, SVM={best['svm_threshold']}$)", fontsize=20, y=1.05)
plt.tight_layout()
plt.show()

top_10_f2 = df_full.sort_values(by="f2", ascending=False).head(10)

# Display the top 10 results
print("--- Top 10 Configurations ---")
print(top_10_f2.to_string(index=False))

In [ ]:
# 3. Filter the full pipeline to just the BEST SVM threshold
df_svm_filtered = df_full[df_full['svm_threshold'] == best['svm_threshold']]

merged_df = pd.merge(
    df_yolo_only, 
    df_svm_filtered, 
    on='yolo_conf', 
    suffixes=('_YOLO_Only', '_with_SVM')
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Precision Comparison (Does the SVM remove false positives?)
axes[0].plot(merged_df['yolo_conf'], merged_df['precision_YOLO_Only'], marker='o', linestyle='--', color='red', label='YOLO Only')
axes[0].plot(merged_df['yolo_conf'], merged_df['precision_with_SVM'], marker='o', color='green', label=f'YOLO + SVM ({best['svm_threshold']})')
axes[0].set_title('Precision Comparison (Higher is less noise)')
axes[0].set_xlabel('YOLO Confidence Threshold')
axes[0].set_ylabel('Precision')
axes[0].legend()
axes[0].grid(True)

# Plot 2: Recall Comparison (Does the SVM accidentally delete real notches?)
axes[1].plot(merged_df['yolo_conf'], merged_df['recall_YOLO_Only'], marker='o', linestyle='--', color='red', label='YOLO Only')
axes[1].plot(merged_df['yolo_conf'], merged_df['recall_with_SVM'], marker='o', color='green', label=f'YOLO + SVM ({best['svm_threshold']})')
axes[1].set_title('Recall Comparison (Higher is fewer missed notches)')
axes[1].set_xlabel('YOLO Confidence Threshold')
axes[1].set_ylabel('Recall')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()